#session spark


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *


In [ ]:
spark = (SparkSession
    .builder
    .appName("skew_cache") \
    .getOrCreate()
)

In [ ]:

# Lectura de datos
Flight_csv = spark.read.csv(
    "/content/drive/flight data.csv",
    header = True,
    inferSchema = True
)


In [ ]:
Flight_csv.rdd.getNumPartitions()


2

In [ ]:
Flight_csv.rdd.glom().map(len).collect()

[308103, 293508]

In [ ]:
# Configuraciones de mitigación para Skew Data
spark.conf.set("spark.sql.adaptive.enable", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewdPartitionFactor", "5")
spark.conf.set("spark.sql.adaptive.skewJoin.skewdPartitionThresholdInBytes", "256mb")

In [ ]:
## Lectura de datos
Flight_csv = spark.read.csv(
    "/content/drive/flight data.csv",
    header = True,
    inferSchema = True
)


In [ ]:
# Número de particiones
Flight_csv.rdd.getNumPartitions()

2

In [ ]:
# Ver tamaño de partición

Flight_csv.rdd.glom().map(len).collect()


[308103, 293508]

In [ ]:

Flight_csv.groupBy("from_airport_code")\
          .count()\
          .orderBy(desc("count"))\
          .show(20)

+-----------------+-----+
|from_airport_code|count|
+-----------------+-----+
|              YYZ|55056|
|              BRU|48776|
|              VIE|48284|
|              SYD|47091|
|              MEL|44636|
|              BOG|43324|
|              CPH|38703|
|              SCL|37025|
|              GRU|33280|
|              CNF|30220|
|              PVG|25667|
|              AEP|21483|
|              CAN|17082|
|              SHA|16527|
|              ALG|15737|
|              CTU|14334|
|              SZX|14174|
|              VCP|12479|
|              XIY|10640|
|              HGH|10358|
+-----------------+-----+
only showing top 20 rows


In [ ]:
# Ver tamaño real de partición tras una repartición mediante esa columna

Flight_airpot_code = Flight_csv.repartition("from_airport_code")
Tamaños = Flight_airpot_code.rdd.glom().map(len).collect()
print(sorted(Tamaños, reverse=True)[:10])


[300520, 257767, 43324]


In [ ]:

# Ejercicio de cache y persist

## Procesamiento con flight

Flight_val = (
    Flight_csv
    .filter(col("stops") == 0)
    .filter(col("co2_percentage") != "0%")
)

Flight_val.persist(StorageLevel.MEMORY_AND_DISK)
Flight_val.count() # Materializa el cache (acción "barata de calentamiento")

# Reporte 1: Precio promedio por país de destino

reporte_precio = (
    Flight_val
    .groupBy("dest_country")
    .agg(avg("price").alias("Precio_promedio"))
)

reporte_precio.show()

NameError: name 'StorageLevel' is not defined

In [ ]:

# Reporte 2: Escritura final

(Flight_val
 .write
 .mode("overwrite")
 .partitionBy("from_country")
 .parquet("salida2/vuelos_validos"))

Flight_val.unpersist()


DataFrame[from_airport_code: string, from_country: string, dest_airport_code: string, dest_country: string, aircraft_type: string, airline_number: string, airline_name: string, flight_number: string, departure_time: timestamp, arrival_time: timestamp, duration: int, stops: int, price: double, currency: string, co2_emissions: int, avg_co2_emission_for_this_route: int, co2_percentage: string, scan_date: timestamp]